In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

In [2]:
tqdm.pandas()

skipping gibberrish data injection for now

In [3]:
# # Injecting synthetic gibberish data into the dataframe so the model learns it
# gibberish_data = pd.DataFrame({
#     'query': ['asdfghjkl', 'qwertyuiop', 'zxcvbnm', 'mnbvcxz', 'lkjhgfdsa', 'poiuytrewq', 'qazwsxedc', 'plmoknijb'],
#     'label': [1, 1, 1, 1, 1, 1, 1, 1]
# })

# # Appending the gibberish to the main dataset
# df = pd.concat([df, gibberish_data], ignore_index=True)

In [4]:
df=pd.read_csv("../data/cleaned/filtered_spam_data.csv")

In [5]:
df.head()

,source_df,text,label,transformed_text
0,df1,Subject: enron methanol ; meter # : 988291\r\n...,0,enron methanol meter 988291 follow note gave m...
1,df1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0,hpl nom januari 9 2001 see attach file hplnol ...
2,df1,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0,neon retreat ho ho ho around wonder time year ...
3,df1,"Subject: photoshop , windows , office . cheap ...",1,photoshop window offic cheap main trend abas d...
4,df1,Subject: re : indian springs\r\nthis deal is t...,0,indian spring deal book teco pvr revenu unders...


In [6]:
df['transformed_text']=df['transformed_text'].astype(str)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=3000)

In [8]:
X = tfidf.fit_transform(df['transformed_text']).toarray()

In [9]:
X.shape

(129565, 3000)

In [10]:
y = df['label'].values

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [13]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [14]:
gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()
dtc = DecisionTreeClassifier(max_depth=5)
svc = SVC(kernel='sigmoid', gamma=1.0, verbose=True)
lrc = LogisticRegression(solver='liblinear', penalty='l1', verbose=1)
rfc = RandomForestClassifier(n_estimators=50, random_state=2, verbose=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2, verbose=1)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2, verbose=2)
gbdt = GradientBoostingClassifier(n_estimators=50, random_state=2, verbose=1)
xgb = XGBClassifier(n_estimators=50, random_state=2, verbosity=2)
knc = KNeighborsClassifier(n_jobs=-1)

In [15]:
clfs = {
    'SVC' : svc,
    'KN' : knc, 
    'MNB': mnb, 
    'GNB': gnb, 
    'BNB': bnb, 
    'DT': dtc, 
    'LR': lrc, 
    'RF': rfc, 
    'AdaBoost': abc, 
    'BgC': bc, 
    'ETC': etc,
    'GBDT':gbdt,
    'xgb':xgb
}

In [16]:
def train_classifier(clf,X_train,y_train,X_test,y_test):
    clf.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    
    return accuracy,precision

In [ ]:
accuracy_scores = []
precision_scores = []

for name, clf in tqdm(clfs.items(), desc="Training Classifiers"):
    
    current_accuracy, current_precision = train_classifier(clf, X_train, y_train, X_test, y_test)
    
    print(f"For {name}")
    print(f"Accuracy - {current_accuracy}")
    print(f"Precision - {current_precision}")
    print("-" * 30)  # Optional: Adds a visual separator between prints
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)

Training Classifiers:   0%|          | 0/13 [00:00<?, ?it/s]

[LibSVM]

In [ ]:
performance_df = pd.DataFrame({'Algorithm':clfs.keys(),'Accuracy':accuracy_scores,'Precision':precision_scores}).sort_values('Precision',ascending=False)

In [ ]:
performance_df

In [ ]:
import pickle
model_name = performance_df['Algorithm'].iloc[0]
with open(f'../models/spam_detection_model/vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open(f'../models/spam_detection_model/spam_model_{model_name}.pkl', 'wb') as f:
    pickle.dump(clfs[model_name], f)

In [ ]:
import nltk
from nltk.corpus import stopwords
import string
nltk.download('punkt')
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
stop_words_set = set(stopwords.words('english'))
punctuation_set = set(string.punctuation)

In [ ]:
def transform_text(text):
    text = text.lower()
    if text.startswith("subject"):
        text = text.replace("subject", "", 1).strip()
    text = nltk.word_tokenize(text)

    y = [
        ps.stem(word) 
        for word in text 
        if word.isalnum() and word not in stop_words_set and word not in punctuation_set
    ]
            
    return " ".join(y)

In [ ]:
def interactive_spam_classifier(model, vectorizer):
    print("\n" + "="*50)
    print("      LIVE EMAIL SPAM CLASSIFIER ACTIVATED      ")
    print("  Type your email text below. Type 'quit' to exit.  ")
    print("="*50 + "\n")
    
    while True:
        # 1. Capture user input
        user_input = input("Enter email text: ").strip()
        
        # 2. Check for exit condition
        if user_input.lower() == 'quit':
            print("\nExiting Classifier. Have a great day!")
            break
            
        if not user_input:
            print("Input cannot be empty. Please try again.\n")
            continue
            
        # 3. Clean the input text using your existing transform_text logic
        cleaned_text = transform_text(user_input)
        
        # 4. Convert the text into a TF-IDF vector matrix matching your 3000 max_features
        vectorized_text = vectorizer.transform([cleaned_text]).toarray()
        
        # 5. Predict using your high-precision Support Vector Classifier
        prediction = model.predict(vectorized_text)[0]
        
        # 6. Output the classification result clearly
        print("-"*40)
        if prediction == 1:
            print("🚨 RESULT: ALERT! This email is classified as SPAM.")
        else:
            print("✅ RESULT: Clean! This email is classified as HAM (Normal).")
        print("-"*40 + "\n")


In [ ]:

# Start the interactive user terminal session immediately
interactive_spam_classifier(svc, tfidf)

